# Gold labeling — manual review of silver labels

In [2]:
import pandas as pd

In [ ]:
SILVER_CSV      = '../data\silver_labels\silver_labels_v1_2026_06_21.csv'
CLEAN_DATA_CSV = '../data/alliance/MDM_Population_cleaned_v5_2026_06_21.parquet'

In [26]:
# Read silver labels. get_silver_labels.ipynb wrote with the index, so drop any
# unnamed index column and force the PATID columns to string (ID dtype trap).
silver = pd.read_csv(SILVER_CSV)

In [67]:
records = pd.read_parquet(CLEAN_DATA_CSV)
records['BirthDT_clean'] = records['BirthDT_clean'].dt.strftime('%Y-%m-%d')

In [68]:
pairs = silver.merge(records.rename({c: c + '_A' for c in records.columns}, axis=1), 
             how='left', on='PATID_A')
pairs = pairs.merge(records.rename({c: c + '_B' for c in records.columns}, axis=1), 
             how='left', on='PATID_B')

In [69]:
pairs['gold_label'] = pd.NA

In [70]:
def show_pairs(df, k=5, fields=FIELDS):
    rows = []
    for _, r in df.head(k).iterrows():
        rows.append({'side':'A','PATID':r['PATID_A'], 'match_prob':round(r['match_prob'],4),
                     'scenario':r.get('scenario',''), **{f:r.get(f'{f}_A') for f in fields}})
        rows.append({'side':'B','PATID':r['PATID_B'], 'match_prob':'',
                     'scenario':'', **{f:r.get(f'{f}_B') for f in fields}})
        rows.append({'side':'', 'PATID':''})  # spacer
    return pd.DataFrame(rows).fillna('')

In [3]:
ALL_FIELDS = ['FirstNM_raw', 'LastNM_raw', 'MiddleNM_raw', 'SuffixNM_raw',
       'BirthDT_raw', 'SSN_raw', 'AddressLine1_raw', 'AddressLine2_raw',
       'CityNM_raw', 'ZipCD_raw', 'StateCD_raw', 'CountryNM',
       'PrimaryPhoneNBR_raw', 'Phone01NBR_raw', 'Phone02NBR_raw',
       'Phone03NBR_raw', 'Email_raw', 'SexAtBirthDSC_raw', 'FirstNM_clean',
       'LastNM_clean', 'MiddleNM_clean', 'SuffixNM_clean', 'BirthDT_clean',
       'SSN_clean', 'last_4_SSN', 'AddressLine1_clean', 'AddressLine2_clean',
       'CityNM_clean', 'ZipCD_clean_base', 'ZipCD_clean_ext', 'StateCD_clean',
       'PrimaryPhoneNBR_clean', 'Phone01NBR_clean', 'Phone02NBR_clean',
       'Phone03NBR_clean', 'Email_clean', 'SexAtBirthDSC_clean',
       'full_name_tokens', 'full_name_compact', 'Phones_set', 'Address_normalized']

CLEAN_FIELDS = ['FirstNM_clean',
       'LastNM_clean', 'MiddleNM_clean', 'SuffixNM_clean', 'BirthDT_clean',
       'SSN_clean', 'last_4_SSN', 'AddressLine1_clean', 'AddressLine2_clean',
       'CityNM_clean', 'ZipCD_clean_base', 'ZipCD_clean_ext', 'StateCD_clean',
       'Email_clean', 'SexAtBirthDSC_clean', 'Phones_set']

In [4]:
def show_pairs(df, k=50, fields=CLEAN_FIELDS):
    rows = []
    pairs = df.head(k).reset_index(drop=True)
    for _, r in pairs.iterrows():
        rows.append({'silver_label': r['silver_label'], **{f:r.get(f'{f}_A') for f in fields}})
        rows.append({'silver_label': r['silver_label'], **{f:r.get(f'{f}_B') for f in fields}})
        rows.append({'silver_label':''})  # spacer

    disp = pd.DataFrame(rows)

    def is_missing(v):
        return v is None or (isinstance(v, float) and pd.isna(v)) or (isinstance(v, str) and v.strip() == '')

    # color matrix matching disp shape
    colors = pd.DataFrame('', index=disp.index, columns=disp.columns)
    for i, r in pairs.iterrows():
        a_idx, b_idx = i * 3, i * 3 + 1  # display rows for this pair
        c = '#00C853' if r['silver_label'] else '#D50000'
        style = f'background-color: {c}'
        colors.loc[a_idx, 'silver_label'] = style
        colors.loc[b_idx, 'silver_label'] = style
        for f in fields:
            a, b = r.get(f'{f}_A'), r.get(f'{f}_B')
            am, bm = is_missing(a), is_missing(b)
            if am and bm:
                c = '#d3d3d3'  # grey  - both missing
            elif am or bm:
                c = '#fff3a3'  # yellow - one missing
            elif isinstance(a, list):
                if len(set(a + b)) == len(a) + len(b):
                    c = '#f4a8a8'  # red    - different
                else:
                    c = '#a8e6a3'  # green  - equal
            elif str(a) == str(b):
                c = '#a8e6a3'  # green  - equal
            else:
                c = '#f4a8a8'  # red    - different
            style = f'background-color: {c}'
            colors.loc[a_idx, f] = style
            colors.loc[b_idx, f] = style
    
    disp = disp.fillna('')

    # strip the _clean suffix on both frames together
    rename_map = {c: c.replace('_clean', '') for c in disp.columns}
    disp = disp.rename(columns=rename_map)
    colors = colors.rename(columns=rename_map)

    return disp.style.apply(lambda _: colors, axis=None)

In [ ]:
show_pairs(pairs, k=5)

In [ ]:
subset = pairs[(pairs['SSN_clean_A']==pairs['SSN_clean_B'])&(pairs['silver_label']==False)]
show_pairs(subset)

## Browser-based gold-labeling app

`labeler.py` runs a tiny local Flask app (127.0.0.1 only). You pass it a **subset** of
`pairs`; it opens a browser tab with a scrollable, color-coded grid. Each pair has
**Match / No match** (→ `gold_label`) and **Ambiguous / OK** (→ `ambiguous_pair`) buttons,
plus keyboard shortcuts (`↑/↓` or `j/k` to move, `1` match, `2` no-match, `3` toggle ambiguous).

Every click writes to a **sparse** `gold_labels.csv` — one row per pair you've *touched*,
keyed by `(PATID_A, PATID_B)`. That CSV is the source of truth and survives restarts; the
notebook reads it back with `load_labels` / `apply_labels`.

Requires `flask` (`pip install flask`). Keep `gold_labels.csv` out of git — it's PHI-derived.

In [10]:
# The app lives in labeler.py (same folder). Re-import freely while iterating.
import importlib, labeler
importlib.reload(labeler)
from labeler import launch_labeler, load_labels, apply_labels, label_summary, stop_labeler

In [11]:
# --- synthetic test set (real data not loadable yet) -----------------------
# Builds a fake `pairs`-shaped DataFrame with the same *_A / *_B columns,
# PATID pairs, silver_label, and a Phones_set list column, so we can exercise
# every cell color (equal / different / one-missing / both-missing / list).
import random
_rng = random.Random(7)

_firsts = ['John', 'Jon', 'Maria', 'María', 'Robert', 'Bob', 'Aisha', 'Wei', 'José', 'Jose']
_lasts  = ['Smith', 'Smyth', 'Garcia', 'García', 'Nguyen', "O'Brien", 'Brown', 'Lee']
_cities = ['Chicago', 'Cicero', 'Evanston', 'Berwyn']
_streets = ['123 Main St', '45 Oak Ave', '900 Halsted St', '12 Lake Shore Dr']

def _mk_record(seed_first, seed_last):
    ssn = f'{_rng.randint(100,899)}-{_rng.randint(10,99)}-{_rng.randint(1000,9999)}'
    return {'FirstNM_clean': seed_first, 'LastNM_clean': seed_last,
            'MiddleNM_clean': _rng.choice([None, 'A', 'Marie', 'Lee']),
            'SuffixNM_clean': _rng.choice([None, None, 'JR']),
            'BirthDT_clean': f'19{_rng.randint(60,99)}-{_rng.randint(1,12):02d}-{_rng.randint(1,28):02d}',
            'SSN_clean': ssn, 'last_4_SSN': ssn[-4:],
            'AddressLine1_clean': _rng.choice(_streets), 'AddressLine2_clean': _rng.choice([None, 'Apt 2']),
            'CityNM_clean': _rng.choice(_cities), 'ZipCD_clean_base': f'606{_rng.randint(10,99)}',
            'ZipCD_clean_ext': _rng.choice([None, '1234']), 'StateCD_clean': 'IL',
            'Email_clean': _rng.choice([None, f'{seed_first.lower()}@mail.com']),
            'SexAtBirthDSC_clean': _rng.choice(['M', 'F']),
            'Phones_set': [f'773555{_rng.randint(1000,9999)}' for _ in range(_rng.randint(1, 2))]}

def _drop_some(rec, p=0.25):
    rec = dict(rec)
    for k in list(rec):
        if k not in ('SSN_clean', 'last_4_SSN') and _rng.random() < p:
            rec[k] = None
    return rec

def make_test_pairs(n=120):
    rows = []
    for i in range(n):
        kind = _rng.choice(['exact', 'variant', 'ssn_diff', 'nomatch', 'missing'])
        a = _mk_record(_rng.choice(_firsts), _rng.choice(_lasts))
        if kind == 'exact':
            b, silver = dict(a), True
        elif kind == 'variant':                      # same person, messy entry
            b = dict(a); b['FirstNM_clean'] = a['FirstNM_clean'][:3]; b['AddressLine1_clean'] = _rng.choice(_streets)
            silver = True
        elif kind == 'ssn_diff':                     # same SSN, different people
            b = _mk_record(_rng.choice(_firsts), _rng.choice(_lasts))
            b['SSN_clean'], b['last_4_SSN'] = a['SSN_clean'], a['last_4_SSN']
            silver = False
        elif kind == 'missing':                      # same person, lots of gaps
            b = _drop_some(a, 0.5); silver = True
        else:                                         # unrelated people
            b = _mk_record(_rng.choice(_firsts), _rng.choice(_lasts)); silver = False
        row = {'PATID_A': f'A{i:05d}', 'PATID_B': f'B{i:05d}',
               'silver_label': silver, 'match_prob': round(_rng.random(), 4), 'scenario': kind}
        row.update({f'{k}_A': v for k, v in a.items()})
        row.update({f'{k}_B': v for k, v in b.items()})
        rows.append(row)
    return pd.DataFrame(rows)

test_pairs = make_test_pairs(120)
test_pairs[['PATID_A', 'PATID_B', 'silver_label', 'scenario']].head()

,PATID_A,PATID_B,silver_label,scenario
0,A00000,B00000,False,ssn_diff
1,A00001,B00001,True,missing
2,A00002,B00002,True,missing
3,A00003,B00003,False,nomatch
4,A00004,B00004,True,exact


In [ ]:
# Launch on the synthetic set (swap test_pairs -> a real subset later, e.g.
#   subset = pairs[(pairs.SSN_clean_A == pairs.SSN_clean_B) & (pairs.silver_label == False)]
#   launch_labeler(subset)
# ). Opens a browser tab; label there, decisions autosave to gold_labels.csv.
launch_labeler(test_pairs[test_pairs.silver_label == False], store_path='gold_labels.csv')

[labeler] running at http://127.0.0.1:8765/
[labeler] store: /Users/miguelroca/Documents/UChicago/Capstone Project/AnyMatch/gold_labeling/gold_labels.csv  (54 pairs loaded)
[labeler] call stop_labeler(8765) to shut it down


'http://127.0.0.1:8765/'

127.0.0.1 - - [23/Jun/2026 16:15:35] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [23/Jun/2026 16:15:36] "GET / HTTP/1.1" 200 -


In [18]:
stop_labeler(8765)

[labeler] stopped server on port 8765


In [14]:
# --- read your labels back into the notebook (run anytime; reflects current CSV) ---
label_summary('gold_labels.csv')          # {'touched':.., 'match':.., 'no_match':.., 'ambiguous':..}

# load_labels('gold_labels.csv')          # the raw store as a DataFrame
# labeled = apply_labels(pairs)           # merge gold_label + ambiguous_pair onto `pairs`
# matches   = load_labels()[lambda d: d.gold_label == 'match'][['PATID_A','PATID_B']]
# nonmatch  = load_labels()[lambda d: d.gold_label == 'no_match'][['PATID_A','PATID_B']]
# stop_labeler()                          # shut the server down when finished

{'touched': 4, 'match': 2, 'no_match': 2, 'ambiguous': 0}